In [17]:
import pandas as pd
import geopandas as gpd
import re
from shapely.geometry import Point

pedestrian_df = pd.read_csv("Bi-Annual_Pedestrian_Counts.csv")
print(pedestrian_df.head())
print(pedestrian_df.info())

                                        the_geom  OBJECTID  Loc Borough  \
0   POINT (-73.90459140730678 40.87919896648574)         1    1   Bronx   
1   POINT (-73.92188432870218 40.82662794123292)         2    2   Bronx   
2   POINT (-73.89535781584335 40.86215460031517)         3    3   Bronx   
3    POINT (-73.87892467324478 40.8812869959873)         4    4   Bronx   
4  POINT (-73.88956389732787 40.844636776717664)         5    5   Bronx   

            Street_Nam         From_Stree          To_Street Iex May07_AM  \
0             Broadway  West 231st Street     Naples Terrace   N    1,189   
1    East 161st Street      Gra Concourse    Sheridan Avenue   Y    1,511   
2    East Fordham Road   Valentine Avenue     Tiebout Avenue   Y    1,832   
3   East Gun Hill Road  Bainbridge Avenue  Rochambeau Avenue   N      764   
4  East Tremont Avenue    Prospect Avenue     Clinton Avenue   N      650   

  May07_PM  ... Oct23_MD June24_AM June24_PM June24_MD Oct24_AM Oct24_PM  \
0    4,094

In [2]:
# Identify and clean time series columns
time_cols = [c for c in pedestrian_df.columns if re.search(r'(May|Sept|Oct|June)\d{2}_(AM|PM|MD)', c, re.IGNORECASE)]
print(time_cols)

['May07_AM', 'May07_PM', 'May07_MD', 'Sept07_AM', 'Sept07_PM', 'Sept07_MD', 'May08_AM', 'May08_PM', 'May08_MD', 'Sept08_AM', 'Sept08_PM', 'Sept08_MD', 'May09_AM', 'May09_PM', 'May09_MD', 'Sept09_AM', 'Sept09_PM', 'Sept09_MD', 'May10_AM', 'May10_PM', 'May10_MD', 'Sept10_AM', 'Sept10_PM', 'Sept10_MD', 'May11_AM', 'May11_PM', 'May11_MD', 'Sept11_AM', 'Sept11_PM', 'Sept11_MD', 'May12_AM', 'May12_PM', 'May12_MD', 'Sept12_AM', 'Sept12_PM', 'Sept12_MD', 'May13_AM', 'May13_PM', 'May13_MD', 'Sept13_AM', 'Sept13_PM', 'Sept13_MD', 'May14_AM', 'May14_PM', 'May14_MD', 'Sept14_AM', 'Sept14_PM', 'Sept14_MD', 'May15_AM', 'May15_PM', 'May15_MD', 'Sept15_AM', 'Sept15_PM', 'Sept15_MD', 'May16_AM', 'May16_PM', 'May16_MD', 'Sept16_AM', 'Sept16_PM', 'Sept16_MD', 'May17_AM', 'May17_PM', 'May17_MD', 'Sept17_AM', 'Sept17_PM', 'Sept17_MD', 'May18_AM', 'May18_PM', 'May18_MD', 'Sept18_AM', 'Sept18_PM', 'Sept18_MD', 'May19_AM', 'May19_PM', 'May19_MD', 'Oct20_AM', 'Oct20_PM', 'Oct20_MD', 'May21_AM', 'May21_PM', 'Ma

In [7]:
def clean_currency(x):
    if isinstance(x, str):
        return pd.to_numeric(x.replace(",", ""), errors='coerce')
    return x

for col in time_cols:
    pedestrian_df[col] = pedestrian_df[col].apply(clean_currency)

print(pedestrian_df.head())

                                        the_geom  OBJECTID  Loc Borough  \
0   POINT (-73.90459140730678 40.87919896648574)         1    1   Bronx   
1   POINT (-73.92188432870218 40.82662794123292)         2    2   Bronx   
2   POINT (-73.89535781584335 40.86215460031517)         3    3   Bronx   
3    POINT (-73.87892467324478 40.8812869959873)         4    4   Bronx   
4  POINT (-73.88956389732787 40.844636776717664)         5    5   Bronx   

            Street_Nam         From_Stree          To_Street Iex  May07_AM  \
0             Broadway  West 231st Street     Naples Terrace   N    1189.0   
1    East 161st Street      Gra Concourse    Sheridan Avenue   Y    1511.0   
2    East Fordham Road   Valentine Avenue     Tiebout Avenue   Y    1832.0   
3   East Gun Hill Road  Bainbridge Avenue  Rochambeau Avenue   N     764.0   
4  East Tremont Avenue    Prospect Avenue     Clinton Avenue   N     650.0   

   May07_PM  ...  Oct23_MD  June24_AM  June24_PM  June24_MD  Oct24_AM  \
0    40

In [8]:
#Calculate monthly totals(AM + PM + MD)

prefixes = set()
for col in time_cols:
    match = re.match(r"((?:May|Sept|Oct|June)\d{2})_(?:AM|PM|MD)", col, re.IGNORECASE)
    if match:
        prefixes.add(match.group(1))

print(prefixes)

{'May12', 'May10', 'May13', 'Oct24', 'Sept09', 'May11', 'Sept18', 'Sept10', 'May09', 'Sept14', 'May23', 'May21', 'Sept12', 'May16', 'Oct23', 'May25', 'May15', 'June24', 'May22', 'Sept13', 'May18', 'Sept16', 'Oct20', 'Oct21', 'Sept08', 'Sept11', 'Sept15', 'May08', 'Sept17', 'Sept07', 'May14', 'May07', 'May19', 'May17', 'Oct22'}


In [10]:
month_year_totals = pd.DataFrame()
for prefix in prefixes:
    relevant_cols = [c for c in time_cols if c.upper().startswith(prefix.upper() + '_')]
    if relevant_cols:
        month_year_totals[prefix] = pedestrian_df[relevant_cols].sum(axis = 1, min_count=1)

print(month_year_totals.head())

     May12    May10    May13    Oct24   Sept09    May11   Sept18   Sept10  \
0   7981.0   7747.0   9745.0   7357.0   7722.0   8124.0   9139.0   7199.0   
1   5020.0   6769.0   8781.0   9356.0   7126.0   6839.0   8333.0   5553.0   
2  16782.0  23652.0  24305.0  12511.0  19656.0  27346.0  17389.0  23179.0   
3   5233.0   8633.0   8012.0   5042.0   6823.0   6015.0   6319.0   5972.0   
4   4501.0  10983.0   8036.0   4942.0   6888.0  11486.0   7197.0  10555.0   

     May09   Sept14  ...   Sept11   Sept15    May08   Sept17   Sept07  \
0   7308.0   9092.0  ...   7576.0   8723.0   7448.0   7886.0   6319.0   
1   5766.0   8772.0  ...   7328.0   9307.0   5296.0  11173.0   6792.0   
2  17535.0  21426.0  ...  24367.0  22391.0  17001.0  22247.0  26109.0   
3   6205.0   7146.0  ...   6419.0   6753.0   7297.0   7433.0   4207.0   
4   5948.0   6569.0  ...  10842.0   6825.0   4643.0   6891.0   6160.0   

     May14    May07    May19    May17    Oct22  
0   8245.0   7791.0      0.0  10585.0   5623.0  


In [12]:
years = set()
for p in prefixes:
    match = re.search(r'\d{2}$', p)
    if match:
        years.add(match.group(0))

sorted_years = sorted(list(years))
print(sorted_years)

['07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25']


In [14]:
year_counts = pd.DataFrame()

for yr in sorted_years:
    cols_for_year = [c for c in month_year_totals.columns if c.endswith(yr)]
    if cols_for_year:
        full_year = f'20{yr}'
        year_counts[full_year] = month_year_totals[cols_for_year].mean(axis=1)

print(year_counts.head())

      2007     2008     2009     2010     2011     2012     2013     2014  \
0   7055.0   8151.5   7515.0   7473.0   7850.0   7920.0   9274.5   8668.5   
1   6729.0   5950.5   6446.0   6161.0   7083.5   6601.0   8287.0   8828.0   
2  27321.5  18760.5  18595.5  23415.5  25856.5  18696.5  22905.0  21477.0   
3   4721.0   7947.0   6514.0   7302.5   6217.0   6362.0   7979.5   6976.5   
4   5841.0   4934.0   6418.0  10769.0  11164.0   5436.5   7544.5   6793.0   

      2015     2016     2017     2018     2019     2020     2021     2022  \
0   9262.0   8122.0   9235.5   8905.5      0.0   5083.0   5723.5   5293.0   
1   9348.5   9613.5  10042.0   8693.0   7625.0   3959.0   6329.5   8694.5   
2  21781.5  22788.0  21714.0  18149.0  20832.0  13006.0  12439.0  12718.5   
3   6892.5   6504.5   6905.5   6091.0      0.0   4201.0   3744.0   4547.5   
4   7072.0   7024.0   7048.5   7279.5      0.0   4073.0   4391.0   4126.0   

      2023     2024     2025  
0   7925.0   6896.0   5455.0  
1   9103.5  

In [15]:
metadata_cols = [c for c in pedestrian_df.columns if c not in time_cols]
processed_df = pd.concat([pedestrian_df[metadata_cols], year_counts],axis=1)

print(processed_df.head())

                                        the_geom  OBJECTID  Loc Borough  \
0   POINT (-73.90459140730678 40.87919896648574)         1    1   Bronx   
1   POINT (-73.92188432870218 40.82662794123292)         2    2   Bronx   
2   POINT (-73.89535781584335 40.86215460031517)         3    3   Bronx   
3    POINT (-73.87892467324478 40.8812869959873)         4    4   Bronx   
4  POINT (-73.88956389732787 40.844636776717664)         5    5   Bronx   

            Street_Nam         From_Stree          To_Street Iex     2007  \
0             Broadway  West 231st Street     Naples Terrace   N   7055.0   
1    East 161st Street      Gra Concourse    Sheridan Avenue   Y   6729.0   
2    East Fordham Road   Valentine Avenue     Tiebout Avenue   Y  27321.5   
3   East Gun Hill Road  Bainbridge Avenue  Rochambeau Avenue   N   4721.0   
4  East Tremont Avenue    Prospect Avenue     Clinton Avenue   N   5841.0   

      2008  ...     2016     2017     2018     2019     2020     2021  \
0   8151.5  .

In [16]:
# Match community districts
cd_map = gpd.read_file("Community_Districts.geojson")
print(cd_map.shape)

(71, 8)


In [20]:
def parse_point(geom_str):
    if pd.isna(geom_str):
        return None
    clean = geom_str.replace("POINT (", "").replace(")", "")
    parts = clean.split()
    return Point(float(parts[0]), float(parts[1]))

processed_df['geometry'] = processed_df['the_geom'].apply(parse_point)

print(processed_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 28 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   the_geom    114 non-null    object 
 1   OBJECTID    114 non-null    int64  
 2   Loc         114 non-null    int64  
 3   Borough     114 non-null    object 
 4   Street_Nam  114 non-null    object 
 5   From_Stree  114 non-null    object 
 6   To_Street   96 non-null     object 
 7   Iex         114 non-null    object 
 8   2007        114 non-null    float64
 9   2008        114 non-null    float64
 10  2009        114 non-null    float64
 11  2010        114 non-null    float64
 12  2011        114 non-null    float64
 13  2012        114 non-null    float64
 14  2013        114 non-null    float64
 15  2014        114 non-null    float64
 16  2015        114 non-null    float64
 17  2016        114 non-null    float64
 18  2017        114 non-null    float64
 19  2018        114 non-null    f

In [23]:
ped_gdf = gpd.GeoDataFrame(processed_df, geometry='geometry', crs="EPSG:4326")

if cd_map.crs != ped_gdf.crs:
    cd_map = cd_map.to_crs[ped_gdf.crs]

joined_gdf = gpd.sjoin(ped_gdf, cd_map, how='left', predicate='within')

def format_cd_id(boro_cd):
    if pd.isna(boro_cd): 
        return None
    boro_cd = str(int(boro_cd))
    boro_map = {'1': 'MN', '2': 'BX', '3': 'BK', '4': 'QN', '5': 'SI'}
    boro_code = boro_cd[0]
    cd_num = boro_cd[1:].zfill(2)
    return boro_map.get(boro_code, '') + cd_num

joined_gdf['Borough_CD'] = joined_gdf['boro_cd'].apply(format_cd_id)
final_columns = [col for col in joined_gdf.columns if col != 'index_right']
final_df = joined_gdf[final_columns].copy()

display_cols = ['Borough_CD'] + list(year_counts.columns) + ['geometry', 'Location', 'Address']
available_cols = [col for col in display_cols if col in final_df.columns]
final_df = final_df[available_cols]
print(final_df)


output_file = 'Pedestrian_Counts_Annual_With_Community_Districts.csv'
final_df.to_csv(output_file, index=False)
print(f"\nResults saved to: {output_file}")


    Borough_CD     2007     2008     2009     2010     2011     2012     2013  \
0         BX08   7055.0   8151.5   7515.0   7473.0   7850.0   7920.0   9274.5   
1         BX04   6729.0   5950.5   6446.0   6161.0   7083.5   6601.0   8287.0   
2         BX05  27321.5  18760.5  18595.5  23415.5  25856.5  18696.5  22905.0   
3         BX07   4721.0   7947.0   6514.0   7302.5   6217.0   6362.0   7979.5   
4         BX06   5841.0   4934.0   6418.0  10769.0  11164.0   5436.5   7544.5   
..         ...      ...      ...      ...      ...      ...      ...      ...   
109       None    788.0    618.0    736.0    822.5    990.5   1064.0    898.5   
110       None    459.0    261.0    370.0    419.0    534.0    639.0    678.5   
111       None    363.0    458.5    263.0    242.5    205.0    370.0    630.0   
112       None     82.5    171.5    182.0    205.5    115.5     75.0     77.0   
113       None    366.5    375.5    468.5   1037.5    677.5    948.5   1140.5   

        2014     2015  ... 